In [ ]:
# Install EasyDiffraction and prepare the example when running in Colab.
import importlib.util
import io
import os
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

if importlib.util.find_spec("edi") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--only-binary=:all:",
        "--extra-index-url", "https://enhantica.github.io/edi/pypi/simple/",
        "easydiffraction",
    ])

if "google.colab" in sys.modules:
    workspace = Path(tempfile.mkdtemp(prefix="edi-tutorial-"))
    (workspace / "notebooks").mkdir()
    os.chdir(workspace / "notebooks")
    target = Path('../cli/pd-neut-cwl_cosio-d20_2k/project').parent.resolve()
    target.mkdir(parents=True, exist_ok=True)
    url = "https://enhantica.github.io/edi/assets/tutorials/pd-neut-cwl_cosio-d20_2k.zip"
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    with zipfile.ZipFile(io.BytesIO(data)) as bundle:
        for member in bundle.infolist():
            if not (target / member.filename).resolve().is_relative_to(target):
                raise ValueError("Unexpected path in tutorial bundle")
        bundle.extractall(target)


In [ ]:
"""Magnetic structure fit: Co₂SiO₄ at 2.3 K, D20."""

# Magnetic structure fit: Co₂SiO₄ at 2.3 K, D20

Fit the magnetic structure of the olivine Co₂SiO₄ to constant-wavelength neutron powder data
from D20 at ILL, measured at 2.3 K, below its ordering temperature. The cobalt moments order with
k = 0 in the magnetic space group Pnma.1: Co1 may point in any direction, Co2 only along b.

The project is the catalogue case `pd-neut-cwl_cosio-d20_2k`: the files the command line fits
with `python -m edi fit docs/user/cli/pd-neut-cwl_cosio-d20_2k/project` and the app opens from
its Examples.

> This tutorial is not run when the documentation is built yet. It loads its project from the edi
> repository, so it runs once a tutorial runner can provide the project elsewhere
> (`execution-exclusions.yml`).

In [ ]:
import edi

## Load the project

In [ ]:
project = edi.Project.load('../cli/pd-neut-cwl_cosio-d20_2k/project')
print(project.metadata.title)
print('Structures:', ', '.join(structure.name for structure in project.structures))
print('Experiments:', ', '.join(experiment.name for experiment in project.experiments))

## The moments

Each magnetic site has one row: its moment's components along the cell axes, in Bohr
magnetons, and the symmetry form the site's symmetry allows. Components the form fixes at zero
cannot be freed.

In [ ]:
for moment in project.structures['cosio'].atom_site_moment:
    print(
        moment.id,
        moment.symmform,
        moment.crystalaxis_x.value,
        moment.crystalaxis_y.value,
        moment.crystalaxis_z.value,
    )

## Fit

The four allowed components are free, with the cell, the coordinates and the profile.

In [ ]:
result = project.analysis.fit()
print(f'{result.status.name.lower()} after {result.iterations} iterations')
print(f'reduced chi-square {result.reduced_chi_square:.3f}, Rwp {100 * result.rwp:.2f} %')

## Fitted moments

Co1's component along c is barely determined by powder data: its uncertainty is as large as the
value.

In [ ]:
for label, value in result.values.items():
    if 'crystalaxis' in label:
        print(f'{label:40} {value:8.3f} ± {result.uncertainty.get(label, 0):.2g}')

## Report and save

The HTML report draws the patterns, and `save_as` writes the fitted project to a folder of its
own; the catalogue's copy stays as it was.

In [ ]:
project.report.save_html('pd-neut-cwl_cosio-d20_2k.html')
project.save_as('pd-neut-cwl_cosio-d20_2k')